### Imports

In [ ]:
from helpers import load_csv_dataset
from pathlib import Path

data_dir = Path('../data')
quotations_dataset = load_csv_dataset(data_dir.joinpath('quotations.csv'), delimiter = ';')

### Concerns

#### 1. Number of indicators per concept

In [ ]:
concepts_temp = []
for quotation in quotations_dataset:
    concepts_temp.extend([x for x in quotation['codes'].split('\n') if 'concept-' in x])
concepts = set(concepts_temp)

In [ ]:
scenarios = ['s1','s2','s3','s4','s5']
concepts_dataset = []
for concept in concepts:
    row = {}
    row['concept'] = concept.replace('concept-','')
    count_s = []
    total = 0
    for s in scenarios:
        count = 0
        for q in quotations_dataset:
            count = count + 1 if concept in q['codes'] and s in q['document'] else count + 0
        row[s] = count
        total += count
    row['total'] = total
    concepts_dataset.append(row)  
concepts_dataset = sorted(concepts_dataset, key=lambda d: d['total'], reverse=True)

In [ ]:
table_data = ''
for c in concepts_dataset:
    row = ''
    for i in c.keys():
        row += str(c[i]) + ' & '
    row += '\\\\\n'
    table_data += row
    
table_data += '\\\hline\\\n'

row = 'total & '
total = 0
for s in scenarios:
    count = 0
    for c in concepts_dataset:
        count += c[s]
    total += count
    row += str(count) + ' & '
    
row += str(total) + ' & '
table_data += row
table_data += '\\\\\hline\\\n'

with open(data_dir.joinpath("table_templates").joinpath("template-number-of-indicators-per-concept.txt"),'r') as f:
    lines = f.readlines()
    id = [x for x, y in enumerate(lines) if y == '\\data\n']
    lines[id[0]] = table_data
    with open(data_dir.joinpath("table_templates").joinpath("number-of-indicators-per-concept.txt"),'w') as f1:
        f1.writelines(lines)
        f1.close()